# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead |Angel Gomez | |
| Member |Kevin Cazzolato |277485|
| Member *(optional)* | Berk Bahar | |


## 2. Mission Title & Research Question

**Title:** *How Trade Policy Uncertainty Regimes Reshape Stock Sector Network Structure*

**Research question:** *How does Trade Policy Uncertainty (TPU) alter the correlation structure and directional interdependencies across US stock market sectors, and which sectors emerge as central nodes or isolated absorbers during high-uncertainty regimes, as identified through the 2025 tariff events?*

**Why it matters:** *Noramlly the TPU is as an isolated risk factor that shifts return distributions or elevates volatility at the individual asset level. This project addresses a structural gap: whether high trade uncertainty actively reconfigures the network of interdependencies between equity sectors, changing not just how much risk exists, but how it propagates. By combining latent regime detection with network analysis, we move beyond unconditional correlations and ask whether trade-sensitive sectors — such as Technology, Industrials, or Consumer Discretionary — systematically become more central or more isolated during periods of elevated policy uncertainty. The answer has direct implications for systemic risk monitoring and regime-aware portfolio construction.*

## 3. Data

**Source(s):**
* **Trade Policy Uncertainty:** Economic Policy Uncertainty (EPU) database (Baker, Bloom, and Davis index) and/or the Federal Reserve Board (Caldara et al. index).
* **Financial Data:** CRSP (Center for Research in Security Prices), Bloomberg, or Yahoo Finance for adjusted daily closing prices and volumes of sector ETFs.
* **Policy Timeline:** Office of the United States Trade Representative (USTR) and the Federal Register for the qualitative timeline of the 2025 tariff events (China, Canada, Mexico).
* **Macro Controls:** FRED (Federal Reserve Economic Data) for the VIX and other broader market indicators.

**Unit of observation:**
The primary unit of observation is the daily trading period. For the network analysis, the cross-sectional units are the 11 US Select Sector SPDR ETFs (e.g., XLK for Technology, XLI for Industrials), observed synchronously over time to capture both contemporaneous and lagged interdependencies.

**Key variables:**
* **State Variable:** The continuous TPU Index, which will be fed into the Hidden Markov Model (HMM) to estimate the latent categorical variable (the regimes, e.g., "Low TPU" vs. "High TPU").
* **Endogenous Network Variables:** Daily log-returns and realized volatilities of the 11 sector ETFs.
* **Derived Network Metrics:** Pairwise correlation and partial correlation matrices computed per regime, node-level centrality measures (degree, betweenness), and pairwise Granger causality statistics to capture directional interdependencies between sectors.
* **Control Variables:** The CBOE Volatility Index (VIX) to capture general market risk aversion, S&P 500 aggregate returns, and interest rate spreads to isolate trade shocks from broad macroeconomic shocks.

**Potential data quality issues:**
* **Frequency Mismatch & Timing:** News-based TPU indices may react with a slight delay or advance compared to financial markets. Aligning daily text-based indices with precise daily market closes requires careful handling to avoid look-ahead bias.
* **Noise in Text-Based Indices:** The Baker, Bloom, and Davis TPU index relies on algorithmic newspaper scraping. This can occasionally generate false positives (e.g., articles discussing historical trade routes rather than current policy uncertainty), introducing noise into the HMM regime estimation.
* **Confounding Macro Shocks:** The 2025 tariff measures likely coincide with other systemic developments (e.g., central bank rate decisions, inflation reports). Failing to properly control for general market volatility (via the VIX) could lead to attributing general market panic exclusively to trade policy.
* **Non-Stationarity:** Both financial time series and continuous indices often exhibit non-stationarity. Careful pre-processing (e.g., utilizing log-returns, first differences, or structural break tests) is required before estimating network metrics to ensure they do not reflect spurious correlations.
* **Look-Ahead Bias & Publication Lag:** Many macroeconomic and policy-based variables are published with a delay relative to the period they describe, or are subsequently revised. For example, the Baker, Bloom, and Davis TPU index is constructed from newspaper archives and may not be available in real time on the reference date; similarly, FRED macro variables such as interest rate spreads are often subject to revision. To prevent data leakage in all supervised learning models, we strictly apply a **publication lag adjustment**: each variable is only made available to the model as of the date it was *publicly released*, not the date it describes. Concretely, this means aligning all features using a conservative lag of at least one trading day between the information release date and its use as a predictor, and never allowing any future information to enter the feature set of a past observation.

In [3]:
import sys
from pathlib import Path

# main.ipynb è nella root della repo → src/ è direttamente sotto
SRC_DIR = Path.cwd() / "src"
sys.path.insert(0, str(SRC_DIR))

from data_acquisition import DataPipeline

pipeline = DataPipeline(start_date="2018-01-01", end_date="2026-06-18")
log_returns, log_prices, macro_raw, tpu_raw, master = pipeline.run_pipeline()

📥 Download ETF prices from Yahoo Finance...
✓ yfinance loaded — 2126 trading days
  Date range  : 2018-01-02 → 2026-06-17
  Price matrix: (2126, 11)
               XLK     XLI     XLY    XLF     XLV    XLE    XLB   XLRE    XLU  \
Date                                                                            
2026-06-15  191.79  178.68  118.57  53.56  152.89  55.55  52.50  44.99  44.74   
2026-06-16  186.44  179.85  118.46  54.35  152.94  55.36  52.72  45.10  45.06   
2026-06-17  185.80  179.60  115.49  54.05  150.71  54.67  52.02  43.97  44.46   

              XLP     XLC  
Date                       
2026-06-15  85.48  112.19  
2026-06-16  85.59  112.32  
2026-06-17  83.68  109.20  

Log-returns : (2009, 11)
             XLK        XLI        XLY        XLF        XLV        XLE  \
count  2009.0000  2009.0000  2009.0000  2009.0000  2009.0000  2009.0000   
mean      0.0009     0.0005     0.0004     0.0004     0.0004     0.0004   
std       0.0168     0.0135     0.0151     0.0148     

### ADF tests — stationarity

- **Log-returns (ETFs)**: all stationary (p-value ≈ 0). Expected — financial returns
  fluctuate around a constant mean with no trend. Ready for models that require
  stationarity (HMM, Granger causality networks, supervised learning).
- **Log-prices (ETFs)**: all show a unit root (high p-value). Expected — price levels
  follow a random walk. Use only for cointegration / VECM analysis, not for models
  that assume stationarity.
- **VIX**: stationary in levels. Plausible — VIX shows volatility clustering but
  reverts to a long-run mean.
- **10y-2y spread, DFF, TPU**: all show a unit root in levels, but are **stationary
  in first differences** (Δ p-value ≈ 0). Classic pattern for I(1) macro/rate series.
  If a stationary feature is needed, use first differences instead of levels.

### Missing values

- Only the `*_rvol` columns show NaNs (~1% of rows), concentrated in the first
  21 trading days: this is the warmup required to compute the 21-day rolling
  volatility. **This is not a data error.**
- For models using the full `master_dataset.csv`, these initial rows will have
  NaNs in the `_rvol` columns — handle with `dropna()` or `fillna()` depending
  on the model.
- For models that depend on realized volatility, start from `warmup_end`
  (printed at runtime) to get the full dataset with no NaNs.
- TPU has 100% coverage over the period, no data availability issues.

The datasets ready in `data/` are:
- `log_returns.csv` → models on stationary returns (HMM, networks, supervised learning)
- `log_prices.csv` → cointegration / VECM
- `realized_vol.csv` → volatility feature (mind the warmup period)
- `master_dataset.csv` → full aligned dataset with 1-day publication lag on macro/TPU



## 4. Planned Methods

## 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [x] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [x] Other: Granger causality networks (pairwise, per regime) with Benjamini-Hochberg FDR correction

*Justification:*  
We will use a causal graph (DAG) to explicitly model the assumed relationships between Trade Policy Uncertainty (TPU), macro-financial conditions, sectoral returns, and inter-sector network structure. The purpose is to formalize identifying assumptions and clarify potential confounders, rather than to claim strong structural causality in a strict econometric sense.

**DAG Structure & Confounders:**  
A central challenge in identifying the causal effect of TPU on sector network structure is the presence of *simultaneous confounders* — variables that independently affect both TPU and sectoral returns, and therefore threaten to produce spurious associations if left unadjusted. In particular:

- The **VIX** (CBOE Volatility Index) reflects broad market risk aversion. A spike in the VIX can simultaneously elevate perceived policy uncertainty *and* compress sector returns across the board, creating a spurious correlation between TPU and network restructuring that is in fact driven by general market panic.
- **Interest rate spreads** capture macroeconomic stress conditions that can coincide with trade policy episodes (e.g., tightening cycles during inflationary periods driven partly by tariff pass-through), again confounding the TPU–sector relationship.

These variables are explicitly represented in the DAG as common causes of both the TPU state variable and the sector return outcomes. Their confounding paths must be blocked before any causal claim about TPU's effect on network structure can be made.

**Backdoor Adjustment:**  
To isolate the exogenous component of TPU shocks from broader market panic, we apply the **backdoor adjustment criterion** (Pearl, 2009). The adjustment set includes the VIX and interest rate spreads, which satisfy the backdoor criterion relative to the TPU → sector network path: they block all non-causal backdoor paths without conditioning on any descendant of TPU. Concretely, all regime-conditional network metrics (correlations, Granger edges, centrality measures) are estimated after partialling out the contemporaneous influence of VIX and interest rate spreads, ensuring that observed network restructuring across TPU regimes reflects genuine trade policy effects rather than confounded macro-financial noise.

**Multiple Testing Correction:**  
Estimating pairwise Granger causality across 11 sectors yields 110 separate statistical tests per regime (11 × 10 directed pairs). At a standard significance level of α = 0.05, approximately 5–6 spurious edges would be expected purely by chance, which could substantially distort the inferred network topology. To control for this, we apply the **Benjamini-Hochberg (BH) False Discovery Rate (FDR)** correction to all p-values within each regime before defining network edges. An edge is retained only if its BH-adjusted p-value falls below the chosen FDR threshold (q = 0.05), ensuring the resulting network reflects statistically robust directional dependencies rather than noise-driven artifacts.

**Limitations:**  
We acknowledge that our DAG encodes assumptions that are not fully testable from observational data alone. In particular, we cannot rule out the existence of unmeasured confounders (e.g., geopolitical sentiment, investor positioning) that lie outside our adjustment set. Results should therefore be interpreted as evidence of robust conditional associations under the stated identifying assumptions, rather than as proof of structural causation.

---

## 4b. Supervised Learning
- [x] Linear / Ridge / Lasso regression
- [x] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Neural network (regression or classification)
- [ ] Other: ___

*Justification:*  
The supervised learning component focuses on predicting directional market outcomes at the sector level, specifically whether future weekly returns are positive or negative.

Logistic regression and regularized linear models (Ridge/Lasso) will serve as baseline models due to their interpretability and robustness in high-noise financial environments. These models will help quantify the marginal predictive contribution of key variables, including TPU regime indicators derived from the HMM, lagged returns, volatility measures, and network centrality features derived from the regime-conditional correlation and Granger networks.

A neural network will be implemented to capture potential nonlinear interactions between TPU regimes and market variables. In particular, the effect of macro uncertainty may be state-dependent — the same signal (e.g., a volatility spike or sector momentum) can have different predictive implications depending on whether the system is in a high- or low-TPU regime. The neural network allows for flexible functional forms to model these interactions, while performance will be carefully benchmarked against simpler models to avoid overfitting.

**Look-Ahead Bias Prevention:**  
A critical requirement for the validity of all supervised learning models is the strict prevention of look-ahead bias. Since our feature set combines financial market data with macroeconomic indices and policy variables — which are often published with a lag or subject to revision — we enforce the following rules throughout the pipeline:

1. **Publication-date alignment:** Every feature is timestamped by its *release date*, not its reference date. For instance, a TPU index value describing week *t* but published on day *t+2* is only used as a predictor from day *t+2* onward.
2. **Forward-only cross-validation:** All time-series cross-validation is performed using an **expanding window** or **walk-forward** scheme, ensuring that the model trained on data up to time *t* is always evaluated on data strictly after *t*. Standard k-fold cross-validation is explicitly excluded, as it allows future observations to contaminate the training set.
3. **Feature construction discipline:** All lagged returns, volatility estimates, network centrality metrics, and HMM regime labels used as features are computed exclusively from information available at prediction time, with no forward-looking smoothing or normalization applied across the full sample.

---

## 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering
- [x] Hierarchical clustering
- [x] Hidden Markov Model (HMM)
- [x] Other: Cointegration analysis (Johansen test / VECM), TVTP-HMM

*Justification:*  
A Hidden Markov Model (HMM) will be used to identify latent regimes of Trade Policy Uncertainty based on the dynamics of the TPU index and associated financial market variables (e.g., volatility and returns). These regimes represent distinct macro-financial states in which the network structure of equity sector interdependencies may differ structurally.

K-Means clustering will be used as a complementary, non-temporal benchmark to group market conditions based on features such as returns, volatility, and volume, providing a simple comparison against the regime structure identified by the HMM.

Hierarchical clustering will further be used to explore whether sectoral behavior naturally groups into stable clusters across time, potentially revealing persistent similarity structures between sectors that are invariant or regime-dependent.

**Time-Varying Transition Probabilities (TVTP-HMM):**  
A standard HMM assumes that the probability of switching between latent regimes is governed by a *fixed* transition matrix, implying that the likelihood of entering a stress regime is constant regardless of prevailing market conditions. This assumption is economically restrictive and empirically implausible.

To relax it, we extend the baseline HMM to a **Time-Varying Transition Probability (TVTP)** specification, following Filardo (1994) and Diebold, Lee & Weinbach (1994). In the TVTP framework, the transition matrix is a function of an observable conditioning variable at each point in time:

$$P(\text{regime}_{t+1} = j \mid \text{regime}_t = i,\, z_t) = \frac{\exp(\alpha_{ij} + \beta_{ij} z_t)}{1 + \exp(\alpha_{ij} + \beta_{ij} z_t)}$$

where $z_t$ is set equal to the **VIX**: spikes in market-implied volatility signal elevated risk aversion, conditions under which trade policy shocks are more likely to trigger or sustain a high-TPU regime. The TVTP-HMM will be evaluated against the standard HMM using AIC/BIC and a log-likelihood ratio test. A positive and significant $\beta$ for the low-to-high transition would confirm that VIX spikes meaningfully increase the probability of entering a high-TPU regime. The resulting smoothed regime probabilities will replace or augment the fixed-transition HMM labels as inputs to the supervised learning models in Section 4b.

**Cointegration Analysis:**  
Working exclusively with log-returns — while necessary to ensure stationarity for short-term network estimation — comes at a cost: differencing prices destroys long-run equilibrium relationships between sectors that may be economically meaningful, particularly during sustained trade policy stress periods.

To recover this long-run dimension, we complement the returns-based analysis with a **cointegration analysis** at the level of log-prices. We apply the **Johansen cointegration test** to the vector of 11 sector ETF log-prices to determine the number of statistically significant cointegrating vectors, reporting both the trace statistic and maximum eigenvalue statistic. Where cointegration is detected, we estimate a **Vector Error Correction Model (VECM)** to jointly capture short-run dynamics and the speed of adjustment toward long-run equilibrium. Both the Johansen test and the VECM are estimated **separately per TPU regime**: a finding that high-TPU regimes exhibit fewer or weaker cointegrating vectors would suggest that sustained trade shocks fracture long-run sectoral equilibria, not just short-term co-movement. The cointegration results serve as a structural complement to the Granger causality networks — while the latter capture directional predictability over short horizons, the VECM captures equilibrium correction dynamics over longer horizons.



## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

The mission will be considered successful if it demonstrates that Trade Policy Uncertainty (TPU) is not only associated with changes in individual sector returns, but also induces a statistically and economically meaningful reconfiguration of the cross-sector correlation and dependency network.

First, we expect the Hidden Markov Model (HMM) to identify distinct and economically interpretable regimes of TPU, where high-uncertainty regimes align with known historical trade policy stress periods. The validity of the regime classification will be assessed by its stability, persistence properties, and its alignment with major trade-related events.

Second, the sectoral network analysis should show clear structural differences across regimes. In particular, we expect that the topology of interdependencies (based on correlation networks and pairwise Granger causality) will differ significantly between low- and high-TPU states. This includes measurable changes in network density, centrality measures, and the identity of sectors occupying central versus peripheral positions.

Third, we aim to identify a subset of sectors that systematically emerge as highly connected nodes during high-TPU regimes. The success of this step will be evaluated by the consistency of these roles across time and their economic interpretability (e.g., trade-exposed sectors such as Industrials or Technology becoming dominant hubs during periods of trade tension).

Finally, predictive performance in supervised learning models should improve when regime-dependent features (HMM states and network centrality metrics) are included, compared to baseline models without regime structure. Even modest but robust improvements would support the hypothesis that TPU-induced regimes contain economically relevant information beyond standard market predictors.

Overall success is defined by convergence of three elements: (1) statistically robust regime identification, (2) regime-dependent restructuring of sectoral networks, and (3) incremental predictive or explanatory power gained by incorporating these regime structures into forecasting models.

## 6. Work Plan

*   **Phase 1: Data Acquisition & Preprocessing** — *Responsible: Kevin*
    *   Collect TPU index data and macro-financial control variables (VIX, interest rate spreads) from FRED and the EPU database.
    *   Retrieve daily adjusted closing prices and trading volumes for the 11 US Select Sector SPDR ETFs.
    *   Preprocess the data: align timestamps to handle trading day discrepancies, apply publication-lag adjustments to macro variables, compute daily log-returns and realized volatilities, and test for stationarity (ADF tests).

*   **Phase 2: Regime Identification & Unsupervised Learning** — *Responsible: Kevin*
    *   Train the standard Hidden Markov Model (HMM) on the continuous TPU index to endogenously classify the market into discrete latent states (Low vs. High TPU regimes).
    *   Extend the HMM to a Time-Varying Transition Probability (TVTP) specification, conditioning the transition matrix on the VIX, and evaluate against the baseline HMM via AIC/BIC and log-likelihood ratio test.
    *   Implement K-Means and Hierarchical clustering as non-temporal benchmarks against the HMM regime structure.
    *   Map the identified high-uncertainty regimes against the qualitative timeline of the 2025 tariff events to validate the model empirically.

*   **Phase 3: Network & Causal Inference Modeling** — *Responsible: Angel*
    *   Formalize the underlying assumptions using a DAG via DoWhy, explicitly mapping confounders (VIX, interest rate spreads) and applying the backdoor adjustment criterion.
    *   Compute regime-conditional correlation and partial correlation matrices for the 11 sector ETFs.
    *   Run pairwise Granger causality tests between all sector pairs separately for each regime, applying Benjamini-Hochberg FDR correction (q = 0.05) before defining network edges.
    *   Analyze structural changes across regimes: compute network-level metrics (density, clustering coefficient) and node-level centrality measures (degree, betweenness).
    * Implement Cointegration Analysis & VECM per TPU regime: Extract log-prices for the 11 sector ETFs, apply the Johansen cointegration test (reporting trace and maximum eigenvalue statistics), and estimate a Vector Error Correction Model (VECM) to quantify the speed of adjustment and assess if sustained trade shocks fracture long-run sectoral equilibria.

*   **Phase 4: Predictive Supervised Learning** — *Responsible: Berk*
    *   Engineer features combining lagged returns, volatility measures, regime indicators (from both standard HMM and TVTP-HMM), and network centrality metrics, with strict publication-date alignment to prevent look-ahead bias.
    *   Train baseline interpretable models (Logistic Regression, Ridge/Lasso) to predict directional sector movements (positive/negative weekly returns).
    *   Design and train the Neural Network to capture non-linear, state-dependent interactions.
    *   Evaluate model performance using Accuracy, F1-score, and ROC-AUC with an expanding-window cross-validation scheme.

*   **Phase 5: 2025 Tariff Case Study & Synthesis** — *Responsible: Berk*
    *   Conduct a deep-dive event study focused on the 2025 tariff announcements regarding China, Canada, and Mexico.
    *   Use the fitted network models to visualize the structural rewiring of sector interdependencies, comparing network topology before and after key tariff announcements.

*   **Phase 6: Finalization & Reporting** — *Responsible: all*
    *   Compile the final Jupyter Notebook with clean, well-documented code.
    *   Generate data visualizations: topological network graphs, regime-conditional correlation heatmaps, HMM regime transition plots, and VECM impulse response functions.
    *   Draft the final academic report synthesizing findings across all three methodological blocks.

## References

- Baker, S. R., Bloom, N., & Davis, S. J. (2016). Measuring economic policy uncertainty. *Quarterly Journal of Economics*, 131(4), 1593–1636.  
  → Primary source for the TPU index used as state variable in the HMM.

- Caldara, D., Iacoviello, M., Molligo, P., Prestipino, A., & Raffo, A. (2020). The economic effects of trade policy uncertainty. *Journal of Monetary Economics*, 109, 38–59.  
  → Direct evidence that TPU has real effects on markets and macro activity.
  
- Filardo, A. J. (1994). Business-cycle phases and their transitional dynamics. *Journal of Business & Economic Statistics*, 12(3), 299–308.  
  → Methodological basis for the TVTP-HMM specification.

---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [6]:
from src.models import HMMRegimeAnalysis

hmm = HMMRegimeAnalysis(smooth_window=10)
regime_df = hmm.run_analysis()


✅ HMM dataset caricato con successo: 2007 obs

═════════════════════════════════════════════════════════════════
ADF STATIONARITY TESTS
═════════════════════════════════════════════════════════════════
  log(TPU) levels           | p=0.0743 | ADF= -2.699 | I(1) ✗  (near-unit-root)
  Δlog(TPU) raw             | p=0.0000 | ADF=-16.613 | I(0) ✓
  VIX levels                | p=0.0000 | ADF= -5.100 | I(0) ✓
═════════════════════════════════════════════════════════════════

Model estimation...
  [log(TPU) univariate | k=2] LL/obs=-2188.483  AIC=8,784,581.1  BIC=8,784,614.7


Model is not converging.  Current: -4301.405985376231 is not greater than -4301.405985145461. Delta is -2.3076972865965217e-07


  [log(TPU)+VIX bivariate | k=2] LL/obs=-4301.406  AIC=17,265,867.6  BIC=17,265,934.9


KeyboardInterrupt: 

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
